# Prospection ProKitchens — API Sirene → Supabase

Ce notebook interroge l'API Sirene de l'INSEE pour trouver des établissements de restauration/traiteurs en Île-de-France, Lyon, Lille et Marseille, puis envoie les résultats dans une table Supabase `leads`.

## 1. Configuration — Identifiants à remplir

In [1]:
# ─── INSEE API (nouvelle authentification par clé API) ───
# Depuis mars 2025, l'ancien OAuth2 (client_credentials) est obsolète.
# Il faut créer une application sur https://portail-api.insee.fr/,
# s'abonner à l'API Sirene, et récupérer la clé API générée.
INSEE_API_KEY = "faf1d66f-9ab7-4986-b1d6-6f9ab7398604"
INSEE_SIRET_URL = "https://api.insee.fr/api-sirene/3.11/siret"

# ─── Supabase ───
SUPABASE_URL = "https://hxjryfaakdpwfgseirik.supabase.co"
SUPABASE_API_KEY = "sb_publishable_a7xpn8srwByQrW1rXwpdlw_eQnwAoHT"
SUPABASE_TABLE = "leads"

## 2. Imports et helpers

In [2]:
import requests
import pandas as pd
import time
import math

## 3. Test de la clé API INSEE

In [3]:
def test_insee_auth():
    """Vérifie que la clé API fonctionne en requêtant un SIRET connu."""
    resp = requests.get(
        f"{INSEE_SIRET_URL}/67205008502051",
        headers={"X-INSEE-Api-Key-Integration": INSEE_API_KEY, "Accept": "application/json"},
    )
    print(f"HTTP {resp.status_code}")
    if resp.status_code == 200:
        nom = resp.json().get("etablissement", {}).get("uniteLegale", {}).get("denominationUniteLegale", "?")
        print(f"✓ Clé API valide — test SIRET : {nom}")
    else:
        print(f"✗ Erreur : {resp.text[:300]}")
        raise RuntimeError("Clé API INSEE invalide ou expirée.")

test_insee_auth()

HTTP 200
✓ Clé API valide — test SIRET : CARREFOUR FRANCE


## 4. Recherche des établissements par zone

In [ ]:
CODES_NAF = ["56.10A", "56.10C", "56.21Z", "56.29B"]

ZONES = {
    "Île-de-France": ["75", "77", "78", "91", "92", "93", "94", "95"],
    "Lyon":          ["69"],
    "Lille":         ["59", "62"],
    "Marseille":     ["13", "83", "84"],
}

PAGE_SIZE = 1000


def build_query_dept(departement):
    naf_filter = " OR ".join(f"activitePrincipaleEtablissement:{c}" for c in CODES_NAF)
    return f"periode({naf_filter}) AND etatAdministratifEtablissement:A AND codePostalEtablissement:{departement}*"


def parse_etablissement(e, zone_name):
    adresse = e.get("adresseEtablissement", {})
    periode = e.get("periodesEtablissement", [{}])[0]
    unite = e.get("uniteLegale", {})
    denomination = (
        periode.get("enseigne1Etablissement")
        or unite.get("denominationUniteLegale")
        or f"{unite.get('prenomUsuelUniteLegale', '')} {unite.get('nomUniteLegale', '')}".strip()
        or "Inconnu"
    )
    return {
        "nom": denomination,
        "ville": adresse.get("libelleCommuneEtablissement", ""),
        "code_postal": adresse.get("codePostalEtablissement", ""),
        "zone": zone_name,
        "siren": e.get("siren", ""),
        "code_naf": periode.get("activitePrincipaleEtablissement", ""),
    }


def fetch_department(dept, zone_name):
    headers = {
        "X-INSEE-Api-Key-Integration": INSEE_API_KEY,
        "Accept": "application/json",
    }
    query = build_query_dept(dept)
    curseur = "*"
    rows = []
    page = 0

    while True:
        page += 1
        params = {"q": query, "nombre": PAGE_SIZE, "curseur": curseur}

        for attempt in range(3):
            resp = requests.get(INSEE_SIRET_URL, headers=headers, params=params)
            if resp.status_code == 429:
                wait = 10 * (attempt + 1)
                print(f"  ⏳ Rate limit — pause {wait}s…")
                time.sleep(wait)
                continue
            break

        if resp.status_code == 404:
            break
        if resp.status_code != 200:
            print(f"  ✗ Dept {dept} — HTTP {resp.status_code} : {resp.text[:200]}")
            break

        data = resp.json()
        etabs = data.get("etablissements", [])
        header = data.get("header", {})
        total = header.get("total", 0)

        rows.extend(parse_etablissement(e, zone_name) for e in etabs)
        print(f"  Dept {dept} — page {page} : {len(rows)}/{total}")

        next_cursor = header.get("curseurSuivant")
        if not next_cursor or next_cursor == curseur or len(rows) >= total:
            break
        curseur = next_cursor
        time.sleep(2)

    return rows


all_rows = []
for zone_name, deps in ZONES.items():
    print(f"\n══ Zone : {zone_name} ══")
    for dept in deps:
        dept_rows = fetch_department(dept, zone_name)
        all_rows.extend(dept_rows)
        print(f"  → Dept {dept} terminé : {len(dept_rows)} établissements")
        time.sleep(2)

print(f"\n══ Total brut : {len(all_rows)} établissements ══")

## 5. Déduplication par SIREN

In [5]:
df = pd.DataFrame(all_rows)
avant = len(df)
df = df.drop_duplicates(subset="siren", keep="first")
print(f"Déduplication : {avant} → {len(df)} (supprimé {avant - len(df)} doublons)")
df.head(10)

Déduplication : 319077 → 265062 (supprimé 54015 doublons)


,nom,ville,code_postal,zone,siren,code_naf
0,COMPAGNIE BOURGUIGNONNE DES OENOPHILES,PARIS 9,75009,Île-de-France,017050402,56.10A
2,LA GAULOISE,PARIS 1,75001,Île-de-France,035520899,56.10A
3,CHAMPAGNE PERRIER-JOUET,PARIS,75006,Île-de-France,095750261,56.10C
4,MAMA GUSTO,PARIS,75011,Île-de-France,100014224,56.10C
5,BELLA PIZZA77,PARIS,75012,Île-de-France,100019264,56.10C
6,VANESSA BOURSEAU,PARIS,75008,Île-de-France,100021757,56.21Z
7,CM FOOD PARIS,PARIS,75008,Île-de-France,100029982,56.21Z
8,VICTOR HUGO,PARIS,75009,Île-de-France,100035500,56.10A
9,CROQ,PARIS,75017,Île-de-France,100039767,56.10C
10,L'ESPERANCE,PARIS,75013,Île-de-France,100046952,56.10A


## 6. Envoi vers Supabase (par lots de 50)

In [6]:
BATCH_SIZE = 50

records = df.copy()
records["statut"] = "nouveau"
payload = records.to_dict(orient="records")

headers = {
    "apikey": SUPABASE_API_KEY,
    "Authorization": f"Bearer {SUPABASE_API_KEY}",
    "Content-Type": "application/json",
    "Prefer": "return=minimal,resolution=merge-duplicates",
}

url = f"{SUPABASE_URL.rstrip('/')}/rest/v1/{SUPABASE_TABLE}?on_conflict=siren"
total_batches = math.ceil(len(payload) / BATCH_SIZE)
inserted = 0

for i in range(0, len(payload), BATCH_SIZE):
    batch_num = i // BATCH_SIZE + 1
    batch = payload[i : i + BATCH_SIZE]
    print(f"  Lot {batch_num}/{total_batches} ({len(batch)} lignes)… ", end="")

    resp = requests.post(url, json=batch, headers=headers)

    if resp.status_code in (200, 201):
        inserted += len(batch)
        print("✓")
    else:
        print(f"✗ HTTP {resp.status_code} — {resp.text[:200]}")

print(f"\n══ {inserted}/{len(payload)} leads upsertés vers Supabase ══")

  Lot 1/5302 (50 lignes)… ✓
  Lot 2/5302 (50 lignes)… ✓
  Lot 3/5302 (50 lignes)… ✓
  Lot 4/5302 (50 lignes)… ✓
  Lot 5/5302 (50 lignes)… ✓
  Lot 6/5302 (50 lignes)… ✓
  Lot 7/5302 (50 lignes)… ✓
  Lot 8/5302 (50 lignes)… ✓
  Lot 9/5302 (50 lignes)… ✓
  Lot 10/5302 (50 lignes)… ✓
  Lot 11/5302 (50 lignes)… ✓
  Lot 12/5302 (50 lignes)… ✓
  Lot 13/5302 (50 lignes)… ✓
  Lot 14/5302 (50 lignes)… ✓
  Lot 15/5302 (50 lignes)… ✓
  Lot 16/5302 (50 lignes)… ✓
  Lot 17/5302 (50 lignes)… ✓
  Lot 18/5302 (50 lignes)… ✓
  Lot 19/5302 (50 lignes)… ✓
  Lot 20/5302 (50 lignes)… ✓
  Lot 21/5302 (50 lignes)… ✓
  Lot 22/5302 (50 lignes)… ✓
  Lot 23/5302 (50 lignes)… ✓
  Lot 24/5302 (50 lignes)… ✓
  Lot 25/5302 (50 lignes)… ✓
  Lot 26/5302 (50 lignes)… ✓
  Lot 27/5302 (50 lignes)… ✓
  Lot 28/5302 (50 lignes)… ✓
  Lot 29/5302 (50 lignes)… ✓
  Lot 30/5302 (50 lignes)… ✓
  Lot 31/5302 (50 lignes)… ✓
  Lot 32/5302 (50 lignes)… ✓
  Lot 33/5302 (50 lignes)… ✓
  Lot 34/5302 (50 lignes)… ✓
  Lot 35/5302 (50 ligne